# 03. Експерименти

## Експериментальний протокол

| | Зміна | Значення | Перевірки |
|---|---|---|---|
| **E1** | тип моделі | AE, VAE | AE краще відновлює, але не генерує з N(0, I) |
| **E2** | розмір латенту d | 2, 4, 8, 16, 32, 64 | компроміс якість / KL; posterior collapse при великих d |
| **E3** | β (вага KL) | 0.1, 0.5, 1, 2, 4, 8 | rate–distortion: відновлення проти «гладкості» латентного простору |
| **E4** | архітектура | MLP, conv | чи дають згортки перевагу |
| **E5** | частка train | 2, 5, 10, 25, 50, 100 % | вибіркова ефективність |
| **E6** | CVAE, d | 2, 8, 16 | генерація заданої цифри (cond_acc) |

**Зафіксовано для всіх запусків** (`configs/base.yaml`): розбиття train/val/test (сід 42), Adam lr = 1e-3,
batch 128, **12 000 кроків** (однаковий бюджет, навіть для малої частки даних), conv-архітектура, d = 16, β = 1 —
змінюється лише параметр експерименту.

**Сіди:** 3 на кожну конфігурацію (0, 1, 2) → звітуємо середнє ± std.
Сід визначає ініціалізацію, порядок батчів і (в E5) саму підвибірку.

**Вибір моделі:** найкращий крок за val-loss (кожні 500 кроків). **Тест** — тільки для фінальних метрик.

In [ ]:
import sys
from pathlib import Path


ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir() and (p / "configs").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

%matplotlib inline
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
import pandas as pd
from src.experiment import run_experiment, collect_results, EXPERIMENTS_DIR as EXP_DIR

# QUICK - перевірка працездатності
QUICK = False
OVERRIDES = {"train.steps": 300, "train.eval_every": 100, "eval.n_generated": 2000} if QUICK else None

SHOW = ["run", "status", "best_step", "train_time_s", "test_recon", "test_kl", "test_nelbo",
        "active_units", "fid_prior", "fid_fitted", "ambiguous_share", "cond_acc"]
pd.set_option("display.max_columns", 30)

## E1. AE проти VAE

In [7]:
df = run_experiment(EXP_DIR / "e1_ae_vs_vae.yaml", overrides=OVERRIDES)
df[[c for c in SHOW if c in df]].round(3)

[1/6] ae_conv_z16_s0_c63525 
    status=ok test_nelbo=62.96 fid_prior=1687.20 time=61.6s
[2/6] ae_conv_z16_s1_5879d8 
    status=ok test_nelbo=63.12 fid_prior=2011.68 time=63.0s
[3/6] ae_conv_z16_s2_84c070 
    status=ok test_nelbo=63.17 fid_prior=2131.36 time=64.0s
[4/6] vae_conv_z16_b1_s0_89f89d 
    status=ok test_nelbo=96.47 fid_prior=290.44 time=81.0s
[5/6] vae_conv_z16_b1_s1_182965 
    status=ok test_nelbo=96.44 fid_prior=281.63 time=77.3s
[6/6] vae_conv_z16_b1_s2_b075c6 
    status=ok test_nelbo=96.82 fid_prior=259.32 time=77.6s
Експеримент завершено за 7.2 хв


,run,status,best_step,train_time_s,test_recon,test_kl,test_nelbo,active_units,fid_prior,fid_fitted,ambiguous_share,cond_acc
0,ae_conv_z16_s0_c63525,ok,12000,61.6,62.964,0.000,62.964,NaN,1687.199,449.469,0.840,None
1,ae_conv_z16_s1_5879d8,ok,12000,63.0,63.115,0.000,63.115,NaN,2011.677,426.625,0.925,None
2,ae_conv_z16_s2_84c070,ok,12000,64.0,63.168,0.000,63.168,NaN,2131.365,444.315,0.941,None
3,vae_conv_z16_b1_s0_89f89d,ok,12000,81.0,74.660,21.815,96.474,14.0,290.437,261.412,0.327,None
4,vae_conv_z16_b1_s1_182965,ok,11500,77.3,74.225,22.216,96.441,15.0,281.634,272.381,0.314,None
5,vae_conv_z16_b1_s2_b075c6,ok,11500,77.6,75.748,21.069,96.817,13.0,259.317,239.344,0.304,None


## E2. Розмір латентного простору

In [8]:
df = run_experiment(EXP_DIR / "e2_latent_dim.yaml", overrides=OVERRIDES)
df[[c for c in SHOW if c in df]].round(3)

[1/18] vae_conv_z2_b1_s0_1bd646 
    status=ok test_nelbo=142.76 fid_prior=573.29 time=79.7s
[2/18] vae_conv_z2_b1_s1_f286be 
    status=ok test_nelbo=144.76 fid_prior=659.46 time=79.5s
[3/18] vae_conv_z2_b1_s2_395994 
    status=ok test_nelbo=143.42 fid_prior=692.82 time=79.0s
[4/18] vae_conv_z4_b1_s0_a10d21 
    status=ok test_nelbo=122.08 fid_prior=299.05 time=80.5s
[5/18] vae_conv_z4_b1_s1_2d6c6c 
    status=ok test_nelbo=121.02 fid_prior=277.37 time=79.2s
[6/18] vae_conv_z4_b1_s2_ac65ba 
    status=ok test_nelbo=121.00 fid_prior=267.26 time=80.1s
[7/18] vae_conv_z8_b1_s0_a99c4e 
    status=ok test_nelbo=102.07 fid_prior=212.35 time=79.8s
[8/18] vae_conv_z8_b1_s1_bd5a64 
    status=ok test_nelbo=102.78 fid_prior=280.59 time=80.6s
[9/18] vae_conv_z8_b1_s2_be011f 
    status=ok test_nelbo=102.39 fid_prior=221.54 time=80.2s
[10/18] vae_conv_z16_b1_s0_89f89d (вже є, пропускаю)
[11/18] vae_conv_z16_b1_s1_182965 (вже є, пропускаю)
[12/18] vae_conv_z16_b1_s2_b075c6 (вже є, пропускаю)
[13/

,run,status,best_step,train_time_s,test_recon,test_kl,test_nelbo,active_units,fid_prior,fid_fitted,ambiguous_share,cond_acc
0,vae_conv_z2_b1_s0_1bd646,ok,12000,79.7,135.973,6.789,142.762,2,573.290,600.068,0.224,None
1,vae_conv_z2_b1_s1_f286be,ok,12000,79.5,138.169,6.587,144.756,2,659.459,702.199,0.245,None
2,vae_conv_z2_b1_s2_395994,ok,12000,79.0,136.893,6.529,143.422,2,692.818,683.515,0.269,None
3,vae_conv_z4_b1_s0_a10d21,ok,12000,80.5,111.496,10.580,122.076,4,299.049,293.854,0.173,None
4,vae_conv_z4_b1_s1_2d6c6c,ok,12000,79.2,109.969,11.051,121.019,4,277.371,282.832,0.187,None
5,vae_conv_z4_b1_s2_ac65ba,ok,11000,80.1,110.072,10.925,120.997,4,267.257,264.828,0.181,None
6,vae_conv_z8_b1_s0_a99c4e,ok,12000,79.8,85.250,16.822,102.072,8,212.350,205.211,0.255,None
7,vae_conv_z8_b1_s1_bd5a64,ok,10500,80.6,85.755,17.024,102.779,8,280.594,241.341,0.281,None
8,vae_conv_z8_b1_s2_be011f,ok,12000,80.2,85.407,16.979,102.386,8,221.536,212.493,0.251,None
9,vae_conv_z16_b1_s0_89f89d,ok,12000,81.0,74.660,21.815,96.474,14,290.437,261.412,0.327,None


## E3. β-VAE (вага KL)

In [9]:
df = run_experiment(EXP_DIR / "e3_beta.yaml", overrides=OVERRIDES)
df[[c for c in SHOW if c in df]].round(3)

[1/18] vae_conv_z16_b0.1_s0_8333d8 
    status=ok test_nelbo=110.99 fid_prior=566.11 time=80.7s
[2/18] vae_conv_z16_b0.1_s1_5ddee5 
    status=ok test_nelbo=111.15 fid_prior=539.55 time=80.9s
[3/18] vae_conv_z16_b0.1_s2_8a2663 
    status=ok test_nelbo=111.02 fid_prior=537.58 time=80.5s
[4/18] vae_conv_z16_b0.5_s0_274d45 
    status=ok test_nelbo=98.01 fid_prior=365.08 time=80.9s
[5/18] vae_conv_z16_b0.5_s1_5d442c 
    status=ok test_nelbo=98.33 fid_prior=327.41 time=80.5s
[6/18] vae_conv_z16_b0.5_s2_7c01dc 
    status=ok test_nelbo=98.07 fid_prior=340.66 time=81.2s
[7/18] vae_conv_z16_b1_s0_89f89d (вже є, пропускаю)
[8/18] vae_conv_z16_b1_s1_182965 (вже є, пропускаю)
[9/18] vae_conv_z16_b1_s2_b075c6 (вже є, пропускаю)
[10/18] vae_conv_z16_b2_s0_80deb3 
    status=ok test_nelbo=100.53 fid_prior=247.62 time=80.2s
[11/18] vae_conv_z16_b2_s1_458067 
    status=ok test_nelbo=100.75 fid_prior=203.91 time=80.8s
[12/18] vae_conv_z16_b2_s2_a94b45 
    status=ok test_nelbo=103.41 fid_prior=233.

,run,status,best_step,train_time_s,test_recon,test_kl,test_nelbo,active_units,fid_prior,fid_fitted,ambiguous_share,cond_acc
0,vae_conv_z16_b0.1_s0_8333d8,ok,12000,80.7,64.050,46.943,110.993,16,566.108,406.708,0.486,None
1,vae_conv_z16_b0.1_s1_5ddee5,ok,12000,80.9,63.857,47.290,111.147,16,539.550,409.172,0.495,None
2,vae_conv_z16_b0.1_s2_8a2663,ok,12000,80.5,63.577,47.445,111.022,16,537.576,403.139,0.492,None
3,vae_conv_z16_b0.5_s0_274d45,ok,12000,80.9,68.328,29.678,98.006,16,365.079,311.751,0.390,None
4,vae_conv_z16_b0.5_s1_5d442c,ok,12000,80.5,68.185,30.149,98.334,16,327.412,311.339,0.362,None
5,vae_conv_z16_b0.5_s2_7c01dc,ok,12000,81.2,68.545,29.526,98.071,16,340.656,298.569,0.368,None
6,vae_conv_z16_b1_s0_89f89d,ok,12000,81.0,74.660,21.815,96.474,14,290.437,261.412,0.327,None
7,vae_conv_z16_b1_s1_182965,ok,11500,77.3,74.225,22.216,96.441,15,281.634,272.381,0.314,None
8,vae_conv_z16_b1_s2_b075c6,ok,11500,77.6,75.748,21.069,96.817,13,259.317,239.344,0.304,None
9,vae_conv_z16_b2_s0_80deb3,ok,12000,80.2,85.168,15.361,100.529,12,247.618,246.213,0.262,None


## E4. MLP проти згорткової архітектури

In [10]:
df = run_experiment(EXP_DIR / "e4_architecture.yaml", overrides=OVERRIDES)
df[[c for c in SHOW if c in df]].round(3)

[1/6] vae_mlp_z16_b1_s0_fc1d38 
    status=ok test_nelbo=100.99 fid_prior=268.40 time=61.5s
[2/6] vae_mlp_z16_b1_s1_9256ac 
    status=ok test_nelbo=100.26 fid_prior=292.53 time=58.6s
[3/6] vae_mlp_z16_b1_s2_93850f 
    status=ok test_nelbo=99.77 fid_prior=271.44 time=59.3s
[4/6] vae_conv_z16_b1_s0_89f89d (вже є, пропускаю)
[5/6] vae_conv_z16_b1_s1_182965 (вже є, пропускаю)
[6/6] vae_conv_z16_b1_s2_b075c6 (вже є, пропускаю)
Експеримент завершено за 3.1 хв


,run,status,best_step,train_time_s,test_recon,test_kl,test_nelbo,active_units,fid_prior,fid_fitted,ambiguous_share,cond_acc
0,vae_mlp_z16_b1_s0_fc1d38,ok,12000,61.5,81.608,19.382,100.990,10,268.403,253.844,0.303,None
1,vae_mlp_z16_b1_s1_9256ac,ok,12000,58.6,80.515,19.746,100.261,11,292.533,269.624,0.315,None
2,vae_mlp_z16_b1_s2_93850f,ok,12000,59.3,79.747,20.027,99.774,12,271.437,249.085,0.310,None
3,vae_conv_z16_b1_s0_89f89d,ok,12000,81.0,74.660,21.815,96.474,14,290.437,261.412,0.327,None
4,vae_conv_z16_b1_s1_182965,ok,11500,77.3,74.225,22.216,96.441,15,281.634,272.381,0.314,None
5,vae_conv_z16_b1_s2_b075c6,ok,11500,77.6,75.748,21.069,96.817,13,259.317,239.344,0.304,None


## E5. Вибіркова ефективність (частка train)

In [11]:
df = run_experiment(EXP_DIR / "e5_data_fraction.yaml", overrides=OVERRIDES)
df[[c for c in SHOW if c in df]].round(3)

[1/18] vae_conv_z16_b1_f0.02_s0_570fa9 
    status=ok test_nelbo=120.86 fid_prior=543.14 time=77.6s
[2/18] vae_conv_z16_b1_f0.02_s1_a19c82 
    status=ok test_nelbo=118.94 fid_prior=507.54 time=81.2s
[3/18] vae_conv_z16_b1_f0.02_s2_c1a8a2 
    status=ok test_nelbo=121.76 fid_prior=531.52 time=82.3s
[4/18] vae_conv_z16_b1_f0.05_s0_188d42 
    status=ok test_nelbo=110.50 fid_prior=441.79 time=81.6s
[5/18] vae_conv_z16_b1_f0.05_s1_43d43d 
    status=ok test_nelbo=110.01 fid_prior=461.02 time=84.3s
[6/18] vae_conv_z16_b1_f0.05_s2_c3516e 
    status=ok test_nelbo=108.97 fid_prior=376.27 time=83.6s
[7/18] vae_conv_z16_b1_f0.1_s0_cd8bfc 
    status=ok test_nelbo=105.86 fid_prior=354.79 time=82.5s
[8/18] vae_conv_z16_b1_f0.1_s1_6cfddd 
    status=ok test_nelbo=104.43 fid_prior=359.90 time=83.3s
[9/18] vae_conv_z16_b1_f0.1_s2_a4943e 
    status=ok test_nelbo=106.38 fid_prior=342.24 time=83.8s
[10/18] vae_conv_z16_b1_f0.25_s0_82e667 
    status=ok test_nelbo=99.79 fid_prior=307.98 time=82.9s
[11

,run,status,best_step,train_time_s,test_recon,test_kl,test_nelbo,active_units,fid_prior,fid_fitted,ambiguous_share,cond_acc
0,vae_conv_z16_b1_f0.02_s0_570fa9,ok,1000,77.6,98.511,22.351,120.863,16,543.140,412.517,0.454,None
1,vae_conv_z16_b1_f0.02_s1_a19c82,ok,1000,81.2,97.934,21.003,118.937,16,507.540,423.457,0.429,None
2,vae_conv_z16_b1_f0.02_s2_c1a8a2,ok,1000,82.3,102.706,19.049,121.755,16,531.515,401.276,0.434,None
3,vae_conv_z16_b1_f0.05_s0_188d42,ok,1500,81.6,89.850,20.649,110.499,16,441.792,352.097,0.404,None
4,vae_conv_z16_b1_f0.05_s1_43d43d,ok,1500,84.3,88.912,21.094,110.006,16,461.024,382.661,0.419,None
5,vae_conv_z16_b1_f0.05_s2_c3516e,ok,2000,83.6,87.235,21.738,108.973,14,376.269,338.327,0.365,None
6,vae_conv_z16_b1_f0.1_s0_cd8bfc,ok,3000,82.5,85.375,20.488,105.863,12,354.791,311.973,0.352,None
7,vae_conv_z16_b1_f0.1_s1_6cfddd,ok,3000,83.3,81.663,22.771,104.434,16,359.902,319.208,0.386,None
8,vae_conv_z16_b1_f0.1_s2_a4943e,ok,3000,83.8,86.385,19.996,106.381,11,342.238,304.462,0.344,None
9,vae_conv_z16_b1_f0.25_s0_82e667,ok,6000,82.9,78.024,21.769,99.793,14,307.982,288.768,0.346,None


## E6. Conditional VAE

In [12]:
df = run_experiment(EXP_DIR / "e6_cvae.yaml", overrides=OVERRIDES)
df[[c for c in SHOW if c in df]].round(3)

[1/9] cvae_conv_z2_b1_s0_a083e5 
    status=ok test_nelbo=129.54 fid_prior=218.71 time=89.8s
[2/9] cvae_conv_z2_b1_s1_53ff8e 
    status=ok test_nelbo=129.72 fid_prior=209.74 time=87.5s
[3/9] cvae_conv_z2_b1_s2_2d8da6 
    status=ok test_nelbo=129.62 fid_prior=213.87 time=88.1s
[4/9] cvae_conv_z8_b1_s0_e3522e 
    status=ok test_nelbo=98.00 fid_prior=79.69 time=90.0s
[5/9] cvae_conv_z8_b1_s1_90377f 
    status=ok test_nelbo=96.27 fid_prior=53.73 time=86.2s
[6/9] cvae_conv_z8_b1_s2_0b78a5 
    status=ok test_nelbo=97.34 fid_prior=62.32 time=88.5s
[7/9] cvae_conv_z16_b1_s0_53adae 
    status=ok test_nelbo=95.50 fid_prior=58.30 time=88.5s
[8/9] cvae_conv_z16_b1_s1_3abbcc 
    status=ok test_nelbo=93.81 fid_prior=60.84 time=87.9s
[9/9] cvae_conv_z16_b1_s2_a72b1c 
    status=ok test_nelbo=93.34 fid_prior=51.33 time=87.9s
Експеримент завершено за 13.4 хв


,run,status,best_step,train_time_s,test_recon,test_kl,test_nelbo,active_units,fid_prior,fid_fitted,ambiguous_share,cond_acc
0,cvae_conv_z2_b1_s0_a083e5,ok,12000,89.8,124.594,4.944,129.537,2,218.710,None,0.010,0.998
1,cvae_conv_z2_b1_s1_53ff8e,ok,12000,87.5,124.889,4.828,129.717,2,209.739,None,0.012,0.995
2,cvae_conv_z2_b1_s2_2d8da6,ok,10500,88.1,124.757,4.863,129.621,2,213.866,None,0.014,0.997
3,cvae_conv_z8_b1_s0_e3522e,ok,11500,90.0,83.552,14.444,97.997,8,79.686,None,0.078,0.957
4,cvae_conv_z8_b1_s1_90377f,ok,11500,86.2,81.780,14.489,96.269,8,53.733,None,0.054,0.975
5,cvae_conv_z8_b1_s2_0b78a5,ok,12000,88.5,82.851,14.491,97.342,8,62.315,None,0.053,0.971
6,cvae_conv_z16_b1_s0_53adae,ok,12000,88.5,79.201,16.302,95.503,10,58.297,None,0.061,0.970
7,cvae_conv_z16_b1_s1_3abbcc,ok,12000,87.9,75.710,18.102,93.813,12,60.844,None,0.072,0.962
8,cvae_conv_z16_b1_s2_a72b1c,ok,11000,87.9,75.528,17.817,93.344,12,51.328,None,0.056,0.971


## Підсумок

In [13]:
results = collect_results(OVERRIDES)
print("Всього записів (запуск × експеримент):", len(results), "| унікальних запусків:", results["run"].nunique())
display(results.groupby(["experiment", "status"]).size().unstack(fill_value=0))
diverged = results[results["status"] != "ok"]
if len(diverged):
    print("Розбіжні запуски:")
    display(diverged[["run", "experiment", "best_step"]])

Всього записів (запуск × експеримент): 75 | унікальних запусків: 63


status,ok
experiment,
e1_ae_vs_vae,6
e2_latent_dim,18
e3_beta,18
e4_architecture,6
e5_data_fraction,18
e6_cvae,9
